# L3 — Return model

Run **Runtime → Run all**. The notebook shows the investment summary and scenario table first; the complete ROCE, DuPont, payback and IRR report is expandable underneath.


In [ ]:
# @title Refresh repository and report tools { display-mode: "form" }
from pathlib import Path
import html as html_lib
import os
import subprocess
import sys

from IPython.display import HTML, display

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / "Model").is_dir() and (cwd / "requirements.txt").exists():
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

model_dir = str((repo / "Model").resolve())
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

def _esc(value):
    return html_lib.escape(str(value))

def show_metrics(metrics):
    cards = []
    for label, value, note in metrics:
        cards.append(
            "<div style='background:#10265c;color:#ffffff;border:1px solid #27457f;"
            "padding:14px 16px;border-radius:10px;min-height:92px'>"
            f"<div style='font-size:12px;font-weight:700;letter-spacing:.03em;color:#dbeafe'>{_esc(label)}</div>"
            f"<div style='font-size:25px;font-weight:800;margin:5px 0 2px'>{_esc(value)}</div>"
            f"<div style='font-size:12px;color:#e4ebf7;line-height:1.35'>{_esc(note)}</div></div>"
        )
    display(HTML(
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));"
        "gap:12px;margin:8px 0 18px;font-family:Arial,sans-serif'>" + "".join(cards) + "</div>"
    ))

def show_table(headers, rows):
    head = "".join(
        f"<th style='background:#10265c;color:#ffffff;padding:8px 10px;text-align:left;"
        f"border:1px solid #cbd5e1'>{_esc(h)}</th>" for h in headers
    )
    body = []
    for index, row in enumerate(rows):
        background = "#ffffff" if index % 2 == 0 else "#eef3f9"
        cells = "".join(
            f"<td style='background:{background};color:#0b1f3a;padding:8px 10px;"
            f"border:1px solid #cbd5e1;vertical-align:top'>{_esc(v)}</td>" for v in row
        )
        body.append(f"<tr>{cells}</tr>")
    display(HTML(
        "<div style='overflow-x:auto;margin:6px 0 16px'>"
        "<table style='border-collapse:collapse;width:100%;font-family:Arial,sans-serif;font-size:13px'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    ))

def show_callout(text, tone="blue"):
    palette = {
        "blue": ("#e4ebf7", "#0b1f3a", "#0070c0"),
        "green": ("#e8f5ec", "#14532d", "#157347"),
        "amber": ("#fff4cc", "#5c4300", "#ffc220"),
        "red": ("#fdecec", "#7f1d1d", "#b3261e"),
    }
    bg, fg, rule = palette[tone]
    display(HTML(
        f"<div style='background:{bg};color:{fg};border-left:6px solid {rule};padding:12px 14px;"
        f"margin:6px 0 16px;font-family:Arial,sans-serif;line-height:1.45'><b>Management read:</b> {_esc(text)}</div>"
    ))

def run_text_report(module_path):
    result = subprocess.run([sys.executable, module_path], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    return result.returncode, output

def show_reports(reports):
    sections = []
    failures = []
    for title, module_path in reports:
        returncode, output = run_text_report(module_path)
        if returncode:
            failures.append((title, returncode))
        sections.append(
            "<details style='margin:10px 0;border:1px solid #94a3b8;border-radius:8px;overflow:hidden'>"
            f"<summary style='cursor:pointer;font-weight:700;background:#e4ebf7;color:#0b1f3a;"
            f"padding:11px 13px;font-family:Arial,sans-serif'>Show full model report — {_esc(title)}</summary>"
            f"<pre style='white-space:pre-wrap;overflow:auto;margin:0;background:#0b1f3a !important;"
            f"color:#f8fafc !important;padding:14px;font-size:12px;line-height:1.45;"
            f"font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace'>{_esc(output)}</pre></details>"
        )
    display(HTML("".join(sections)))
    if failures:
        raise RuntimeError("Report failed: " + ", ".join(f"{name} (exit {code})" for name, code in failures))

print(f"Repository ready: {repo}")


In [ ]:
# @title Refresh compact management summary { display-mode: "form" }
import math
import roce as returns

scenarios = returns.scenario_rows()
show_metrics([
    ("Breakeven AOV", f"Rs{returns.AOV_BREAKEVEN:.0f}", "ROCE = 0; this is not the hurdle"),
    ("40% hurdle AOV", f"Rs{returns.AOV_HURDLE:.0f}", "Closed-form solution on the base capital employed"),
    ("ROCE at 30% basket", f"{returns.roce(returns.AOV_UP):.1%}", "Underwritten management scenario"),
    ("ROCE at 40% basket", f"{returns.roce(returns.AOV_MAX):.1%}", "Upper end of the disclosed cross-operator range"),
])
scenario_rows = []
for row in scenarios:
    payback = "not meaningful" if not math.isfinite(row["payback"]) else f"{row['payback']:.1f} months"
    scenario_rows.append([
        row["name"], f"Rs{row['aov']:.0f}", f"{row['volume']:.0f}/day", f"{row['roce']:.1%}", payback
    ])
show_table(["Scenario", "AOV", "Volume", "ROCE", "Payback"], scenario_rows)
show_callout(
    "The 30% basket clears a positive-return pilot case but not the 40% external comparator. The volume-shock row is the stop condition, not a sensitivity to average away.",
    "amber",
)


## How to read the summary

Breakeven and hurdle AOV answer different questions. Read each return together with its basket, throughput, capital-employed and node-life basis.


In [ ]:
# @title Build expandable full report { display-mode: "form" }
show_reports([("return on capital", "Model/roce.py")])


## Open the model and workbook


- [Open the live model workbook in Google Sheets](https://docs.google.com/spreadsheets/d/1Gws3MUD36160MSC_N0Bt9N_92xGAku3_/edit?gid=1304492030#gid=1304492030)
- [Download `Campus_Store_Model.xlsx` from GitHub](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Campus_Store_Model.xlsx?raw=1)
- [Browse the complete public `Model/` folder](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/tree/main/Model)

- [Return-model source](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Model/roce.py)
